In [0]:
mdb = spark.table(
    "`ph-com-ai-us-prod-catalog-2553213839599380`."
    "kerendia_oe_journey."
    "final_mdb_oe_2_0_21_instances"
)

# Check shape
print("Rows:", mdb.count())
print("Columns:", len(mdb.columns))

In [0]:
mdb.limit(5).display()

In [0]:
column_name = "BH_ID"

if column_name in mdb.columns:
    print(f"✅ Column exists: {column_name}")
else:
    print(f"❌ Column does not exist: {column_name}")

In [0]:
mdb.select("instance").distinct().display()

In [0]:
from pyspark.sql.types import (
    StringType, DoubleType, LongType, DateType,
    IntegerType, FloatType, ArrayType, DecimalType
)

In [0]:
from collections import defaultdict
from pyspark.sql import Row
 
# Create dictionary of datatype -> columns
dtype_dict = defaultdict(list)
 
for col, dtype in mdb.dtypes:
    dtype_dict[dtype].append(col)
 
# Convert to list of Rows
data = [
    Row(datatype=dtype, columns=cols)
    for dtype, cols in dtype_dict.items()
]
 
# Create PySpark DataFrame
dtype_df = spark.createDataFrame(data)
 
# Display result
dtype_df.display()

In [0]:
!pip install openpyxl

In [0]:
import pandas as pd

# ============================================================
# Load Excel
# ============================================================

feature_df = pd.read_excel("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_oe_journey/kerendia_oe_volume/oe_git_offload/Kerendia_OE_2.0/Feature Selection/final_feature_list_2k_modelling.xlsx")

# Column containing feature names
excel_features = (
    feature_df["feature"]
    .dropna()
    .astype(str)
    .str.strip()
    .drop_duplicates()
    .tolist()
)

# ============================================================
# Compare against mdb columns
# ============================================================

mdb_cols = set(mdb.columns)

present_features = [
    f for f in excel_features
    if f in mdb_cols
]

missing_features = [
    f for f in excel_features
    if f not in mdb_cols
]

print("Features in Excel       :", len(excel_features))
print("Present in mdb          :", len(present_features))
print("Missing from mdb        :", len(missing_features))

# ============================================================
# Validation
# ============================================================

if len(missing_features) == 0:
    print("✅ All Excel features are present in mdb.")
else:
    print("❌ Some features are missing from mdb.")
    print("Sample missing features:")
    print(missing_features[:20])

In [0]:
import pandas as pd
from pyspark.sql import functions as F

# ============================================================
# Load Excel
# ============================================================

feature_file = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_oe_journey/kerendia_oe_volume/oe_git_offload/Kerendia_OE_2.0/Feature Selection/final_feature_list_2k_modelling.xlsx"

feature_df = pd.read_excel(feature_file)

print("Excel shape:", feature_df.shape)
display(feature_df.head())

# ============================================================
# Get feature names from column 'feature'
# ============================================================

selected_features = (
    feature_df["feature"]
    .dropna()
    .astype(str)
    .str.strip()
    .drop_duplicates()
    .tolist()
)

print("Features from excel:", len(selected_features))

# ============================================================
# Additional columns you want to keep
# Add your own columns here
# ============================================================

additional_cols = [
    "PROVIDER_ID",
    "INSTANCE",
    "TOTAL_NBRX_C4",
    "full_channelName_seq",
         "full_channelCode_seq",
                "channelName_seq",
                "channelCode_seq",
                "full_contentName_seq",
                "full_contentCode_seq",
                "contentName_seq",
                "contentCode_seq",
                "timeSeq",
                "full_channelName_seq_GAP_grp",
                "full_channelCode_seq_GAP_grp",
                "full_contentName_seq_GAP_grp",
                "full_contentCode_seq_GAP_grp"
]

# ============================================================
# Filter only columns that actually exist in mdb
# ============================================================

mdb_cols = set(mdb.columns)

final_cols = [
    c for c in (selected_features + additional_cols)
    if c in mdb_cols
]

# Remove duplicates while preserving order
final_cols = list(dict.fromkeys(final_cols))

print("Final columns selected:", len(final_cols))

missing_features = [
    c for c in selected_features
    if c not in mdb_cols
]

print("Features missing from mdb:", len(missing_features))

# Optional check
if len(missing_features) > 0:
    print("Sample missing features:")
    print(missing_features[:20])

# ============================================================
# Filter mdb
# ============================================================

mdb_filtered = mdb.select(
    *[
        F.col("`" + c.replace("`", "``") + "`").alias(c)
        for c in final_cols
    ]
)

print("Filtered shape:")
print(
    (
        mdb_filtered.count(),
        len(mdb_filtered.columns)
    )
)

display(mdb_filtered.limit(10))

In [0]:
mdb_filtered.write.mode("overwrite").saveAsTable(
    "`ph-com-ai-us-prod-catalog-2553213839599380`."
    "kerendia_oe_journey."
    "final_mdb_oe_2_0_21_instances_filt"
)

In [0]:
column_name = "channelName_seq"

if column_name in mdb_filtered.columns:
    print(f"✅ Column exists: {column_name}")
else:
    print(f"❌ Column does not exist: {column_name}")